In [ ]:
# ============================================================
# ALRIYADH - DIRECT ARTICLE SCRAPER
# PART 03
# ID RANGE: 520,000 -> 729,999
# ============================================================

!pip -q install requests beautifulsoup4 lxml

import os
import time
import warnings
import requests
import pandas as pd

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib3.exceptions import InsecureRequestWarning

from google.colab import drive


# ============================================================
# GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# SETTINGS
# ============================================================

START_ID = 520_000
END_ID = 729_999

PART_NAME = "part_03"

WORKERS = 5

TIMEOUT = 20

SAVE_EVERY = 5_000

CHUNK_SIZE = WORKERS * 10


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

BASE_DIR = "/content/drive/MyDrive/alriyadh_1million"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    PART_NAME
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

PROGRESS_FILE = os.path.join(
    OUTPUT_DIR,
    "progress.txt"
)


# ============================================================
# WARNINGS
# ============================================================

warnings.simplefilter(
    "ignore",
    InsecureRequestWarning
)


# ============================================================
# RESUME SYSTEM
# ============================================================

if os.path.exists(PROGRESS_FILE):

    try:

        with open(
            PROGRESS_FILE,
            "r"
        ) as f:

            saved_id = int(
                f.read().strip()
            )

        CURRENT_ID = saved_id + 1

        print(
            f"RESUMING FROM ID: {CURRENT_ID:,}"
        )

    except Exception:

        CURRENT_ID = START_ID

        print(
            f"STARTING FROM ID: {CURRENT_ID:,}"
        )

else:

    CURRENT_ID = START_ID

    print(
        f"STARTING FROM ID: {CURRENT_ID:,}"
    )


if CURRENT_ID > END_ID:

    print("=" * 80)
    print("THIS PART IS ALREADY FINISHED.")
    print("=" * 80)

    raise SystemExit


# ============================================================
# SESSION
# ============================================================

def create_session():

    session = requests.Session()

    retry = Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=1,

        status_forcelist=[
            429,
            500,
            502,
            503,
            504
        ],

        allowed_methods=[
            "GET"
        ]
    )

    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=WORKERS,
        pool_maxsize=WORKERS
    )

    session.mount(
        "http://",
        adapter
    )

    session.mount(
        "https://",
        adapter
    )

    session.headers.update({

        "User-Agent":
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/151.0.0.0 Safari/537.36",

        "Accept-Language":
            "ar-SA,ar;q=0.9,en;q=0.8"

    })

    return session


# ============================================================
# SCRAPE ONE ARTICLE
# ============================================================

def scrape_article(article_id):

    session = create_session()

    url = f"https://www.alriyadh.com/{article_id}"

    try:

        response = session.get(
            url,
            timeout=TIMEOUT,
            verify=False
        )

        if response.status_code != 200:
            return None


        soup = BeautifulSoup(
            response.text,
            "lxml"
        )


        # ====================================================
        # TITLE
        # ====================================================

        title = ""

        og_title = soup.find(
            "meta",
            property="og:title"
        )

        if (
            og_title
            and og_title.get("content")
        ):

            title = (
                og_title["content"]
                .strip()
            )


        if not title:

            h1 = soup.find("h1")

            if h1:

                title = h1.get_text(
                    " ",
                    strip=True
                )


        # ====================================================
        # CATEGORY
        # ====================================================

        category = ""

        active_items = soup.select(
            "li.active"
        )

        categories = []

        for li in active_items:

            a = li.find("a")

            if not a:
                continue

            text = a.get_text(
                " ",
                strip=True
            )

            if text:

                categories.append(text)


        if len(categories) >= 2:

            category = categories[1]

        elif len(categories) == 1:

            category = categories[0]


        # ====================================================
        # COUNTRY
        # ====================================================

        country = ""

        country_candidates = soup.select(
            '[class*="country"],'
            '[class*="Country"],'
            '[itemprop="contentLocation"]'
        )

        for element in country_candidates:

            text = element.get_text(
                " ",
                strip=True
            )

            if text:

                country = text

                break


        # ====================================================
        # DATE
        # ====================================================

        date = ""

        date_candidates = soup.select(
            '[itemprop="datePublished"],'
            'time,'
            '[class*="date"],'
            '[class*="Date"]'
        )

        for element in date_candidates:

            value = (
                element.get("datetime")
                or element.get_text(
                    " ",
                    strip=True
                )
            )

            if value:

                date = value

                break


        # ====================================================
        # VALID ARTICLE
        # ====================================================

        if not title:

            return None


        return {

            "url": url,

            "title": title,

            "category": category,

            "country": country,

            "date": date

        }


    except Exception:

        return None


# ============================================================
# SAVE BATCH
# ============================================================

def save_batch(
    data,
    batch_number
):

    if not data:
        return


    df = pd.DataFrame(
        data
    )


    output_file = os.path.join(
        OUTPUT_DIR,
        f"batch_{batch_number:05d}.csv"
    )


    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )


    print()
    print("=" * 80)

    print(
        f"SAVED: {output_file}"
    )

    print(
        f"ROWS: {len(df):,}"
    )

    print("=" * 80)


# ============================================================
# FIND NEXT BATCH NUMBER
# ============================================================

existing_batches = [

    filename

    for filename in os.listdir(
        OUTPUT_DIR
    )

    if filename.startswith("batch_")
    and filename.endswith(".csv")

]


batch_number = 1


if existing_batches:

    numbers = []

    for filename in existing_batches:

        try:

            number = int(
                filename
                .replace(
                    "batch_",
                    ""
                )
                .replace(
                    ".csv",
                    ""
                )
            )

            numbers.append(
                number
            )

        except Exception:

            pass


    if numbers:

        batch_number = max(numbers) + 1


# ============================================================
# START
# ============================================================

print()
print("=" * 80)

print(
    "ALRIYADH - DIRECT ARTICLE SCRAPER"
)

print("=" * 80)

print(
    f"PART:       {PART_NAME}"
)

print(
    f"START ID:   {CURRENT_ID:,}"
)

print(
    f"END ID:     {END_ID:,}"
)

print(
    f"WORKERS:    {WORKERS}"
)

print(
    f"SAVE EVERY: {SAVE_EVERY:,}"
)

print(
    f"CHUNK SIZE: {CHUNK_SIZE}"
)

print("=" * 80)


start_time = time.time()

batch = []

completed_ids = CURRENT_ID - 1

current = CURRENT_ID


# ============================================================
# MAIN LOOP
# ============================================================

while current <= END_ID:

    chunk_end = min(
        current + CHUNK_SIZE - 1,
        END_ID
    )


    ids = range(
        current,
        chunk_end + 1
    )


    # ========================================================
    # RUN CHUNK
    # ========================================================

    with ThreadPoolExecutor(
        max_workers=WORKERS
    ) as executor:

        futures = {

            executor.submit(
                scrape_article,
                article_id
            ): article_id

            for article_id in ids

        }


        for future in as_completed(
            futures
        ):

            try:

                result = future.result()

                if result:

                    batch.append(
                        result
                    )

            except Exception:

                pass


    # ========================================================
    # UPDATE PROGRESS
    # ========================================================

    completed_ids = chunk_end


    with open(
        PROGRESS_FILE,
        "w"
    ) as f:

        f.write(
            str(completed_ids)
        )


    # ========================================================
    # SAVE EVERY 5,000 ARTICLES
    # ========================================================

    while len(batch) >= SAVE_EVERY:

        save_data = batch[
            :SAVE_EVERY
        ]

        batch = batch[
            SAVE_EVERY:
        ]

        save_batch(
            save_data,
            batch_number
        )

        batch_number += 1


    # ========================================================
    # PROGRESS
    # ========================================================

    elapsed = (
        time.time()
        - start_time
    )


    checked = (
        completed_ids
        - CURRENT_ID
        + 1
    )


    rate = (

        checked / elapsed

        if elapsed > 0

        else 0

    )


    print(
        f"Progress: "
        f"{completed_ids:,}/"
        f"{END_ID:,} | "
        f"Checked: {checked:,} | "
        f"Collected pending: "
        f"{len(batch):,} | "
        f"Rate: {rate:.2f} IDs/sec"
    )


    # ========================================================
    # NEXT CHUNK
    # ========================================================

    current = chunk_end + 1


# ============================================================
# SAVE REMAINING ARTICLES
# ============================================================

if batch:

    save_batch(
        batch,
        batch_number
    )


# ============================================================
# FINISHED
# ============================================================

elapsed = (
    time.time()
    - start_time
)


print()
print("=" * 80)

print(
    "PART FINISHED"
)

print("=" * 80)

print(
    f"PART:       {PART_NAME}"
)

print(
    f"LAST ID:    {END_ID:,}"
)

print(
    f"TIME:       {elapsed / 3600:.2f} hours"
)

print(
    f"OUTPUT:     {OUTPUT_DIR}"
)

print("=" * 80)

Mounted at /content/drive
RESUMING FROM ID: 521,850

ALRIYADH - DIRECT ARTICLE SCRAPER
PART:       part_03
START ID:   521,850
END ID:     729,999
WORKERS:    5
SAVE EVERY: 5,000
CHUNK SIZE: 50
Progress: 521,899/729,999 | Checked: 50 | Collected pending: 46 | Rate: 4.85 IDs/sec
Progress: 521,949/729,999 | Checked: 100 | Collected pending: 79 | Rate: 4.86 IDs/sec
Progress: 521,999/729,999 | Checked: 150 | Collected pending: 128 | Rate: 4.87 IDs/sec
Progress: 522,049/729,999 | Checked: 200 | Collected pending: 178 | Rate: 4.84 IDs/sec
Progress: 522,099/729,999 | Checked: 250 | Collected pending: 227 | Rate: 4.85 IDs/sec
Progress: 522,149/729,999 | Checked: 300 | Collected pending: 276 | Rate: 4.86 IDs/sec
Progress: 522,199/729,999 | Checked: 350 | Collected pending: 326 | Rate: 4.86 IDs/sec
Progress: 522,249/729,999 | Checked: 400 | Collected pending: 366 | Rate: 4.86 IDs/sec
Progress: 522,299/729,999 | Checked: 450 | Collected pending: 412 | Rate: 4.86 IDs/sec
Progress: 522,349/729,999 

NameError: name 'pending' is not defined